In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Pertemuan12-MLlibKlasifikasi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [10]:
import numpy as np
import pandas as pd

df_tugas12 = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)
np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


In [11]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# 1. StringIndexer untuk kolom kategorikal
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

# 2. OneHotEncoder untuk ketiga kolom indeks
ohe_tugas = OneHotEncoder(
    inputCols=["menu_idx", "cabang_idx", "waktu_idx"],
    outputCols=["menu_vec", "cabang_vec", "waktu_vec"]
)

# 3. VectorAssembler menggabungkan fitur one-hot dengan qty dan waktu_tunggu_menit
assembler_tugas = VectorAssembler(
    inputCols=["qty", "waktu_tunggu_menit", "menu_vec", "cabang_vec", "waktu_vec"],
    outputCol="features"
)

# 4. Algoritma LogisticRegression
lr_tugas = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

# Merangkai seluruh tahap ke dalam Pipeline
pipeline_tugas_lr = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe_tugas, assembler_tugas, lr_tugas])
print("Pipeline Logistic Regression berhasil didefinisikan.")

Pipeline Logistic Regression berhasil didefinisikan.


In [12]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Split data 80:20 dengan seed=77
train_t, test_t = df_tugas12.randomSplit([0.8, 0.2], seed=77)
print(f"Data training: {train_t.count()} | Data testing: {test_t.count()}")

# Melatih pipeline & membuat prediksi pada data testing
model_tugas_lr = pipeline_tugas_lr.fit(train_t)
pred_tugas_lr = model_tugas_lr.transform(test_t)

# Evaluator metrik
ev_acc_t = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1_t = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc_t = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

acc_lr = ev_acc_t.evaluate(pred_tugas_lr)
f1_lr = ev_f1_t.evaluate(pred_tugas_lr)
auc_lr = ev_auc_t.evaluate(pred_tugas_lr)

print("=== Evaluasi Logistic Regression ===")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

# Confusion Matrix sederhana menggunakan groupBy
print("\nConfusion Matrix (Logistic Regression):")
pred_tugas_lr.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

Data training: 2411 | Data testing: 589
=== Evaluasi Logistic Regression ===
Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650

Confusion Matrix (Logistic Regression):
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



In [13]:
from pyspark.ml.classification import RandomForestClassifier

# Mendefinisikan RandomForestClassifier (numTrees=30)
rf_tugas = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30)

# Pipeline baru dengan 5 tahap feature engineering yang identik
pipeline_tugas_rf = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe_tugas, assembler_tugas, rf_tugas])

model_tugas_rf = pipeline_tugas_rf.fit(train_t)
pred_tugas_rf = model_tugas_rf.transform(test_t)

acc_rf = ev_acc_t.evaluate(pred_tugas_rf)
f1_rf = ev_f1_t.evaluate(pred_tugas_rf)
auc_rf = ev_auc_t.evaluate(pred_tugas_rf)

print("=== Perbandingan Logistic Regression vs Random Forest ===")
print(f"Logistic Regression -> Accuracy: {acc_lr:.4f} | F1-score: {f1_lr:.4f} | AUC-ROC: {auc_lr:.4f}")
print(f"Random Forest       -> Accuracy: {acc_rf:.4f} | F1-score: {f1_rf:.4f} | AUC-ROC: {auc_rf:.4f}")

=== Perbandingan Logistic Regression vs Random Forest ===
Logistic Regression -> Accuracy: 0.7929 | F1-score: 0.7926 | AUC-ROC: 0.8650
Random Forest       -> Accuracy: 0.7946 | F1-score: 0.7942 | AUC-ROC: 0.8472


In [15]:
# Memilih prediksi dari model terbaik berdasarkan AUC tertinggi
pred_terbaik = pred_tugas_lr if auc_lr >= auc_rf else pred_tugas_rf
nama_terbaik = "Logistic Regression" if auc_lr >= auc_rf else "Random Forest"
print(f"Model terbaik berdasarkan AUC-ROC: {nama_terbaik}\n")

# Menampilkan 10 baris di mana model salah memprediksi (ulasan_positif != prediction)
pred_terbaik.filter(col("ulasan_positif") != col("prediction")) \
    .select("nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit", "ulasan_positif", "prediction", "probability") \
    .show(10, truncate=False)

Model terbaik berdasarkan AUC-ROC: Logistic Regression

+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253879175414356,0.37461208245856437]|
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |[0.9113822566149019,0.08861774338509809]|
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |[0.23209137939008503,0.7679086206099149]|
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |[0.42615454016906074,0.5738454598309393]|
|Americano|Magelang|Sore       |3  |5                 |0             |1.0       |[0

Berdasarkan hasil pemodelan klasifikasi yang telah dibangun menggunakan data transaksi dan ulasan pelanggan "Kopi Nusantara", variabel yang paling dominan memengaruhi kecenderungan pelanggan dalam memberikan ulasan positif (rating ≥ 4) adalah waktu tunggu penyajian (waktu_tunggu_menit). Hasil analisis kesalahan model memperlihatkan bahwa titik kritis kepuasan pelanggan berada pada ambang batas 12 menit, di mana pesanan yang disajikan di bawah 10 menit memiliki peluang sangat tinggi untuk mendapatkan ulasan positif, sedangkan waktu tunggu yang melampaui 12 hingga 14 menit secara signifikan menurunkan tingkat kepuasan pelanggan terlepas dari jenis menu maupun lokasi cabang pemesanan. Oleh karena itu, manajemen "Kopi Nusantara" direkomendasikan untuk menetapkan standar operasional waktu penyajian maksimal 10 menit, terutama pada jam sibuk seperti siang dan sore hari serta pada menu makanan yang memerlukan waktu pengolahan lebih lama seperti Roti Bakar dan Kentang Goreng. Selain itu, pihak manajemen perlu mengoptimalkan alur kerja dapur dengan menyiapkan bahan setengah jadi sebelum jam ramai, memisahkan jalur antrean minuman cepat saji dari menu makanan, serta menerapkan sistem peringatan dini pada kasir agar staf dapat menginformasikan estimasi waktu tunggu atau memberikan kompensasi kecil kepada pelanggan apabila antrean mulai mendekati ambang batas kritis 12 menit.

In [16]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
